# Занятие 5. Построение и обучение модели

Прогноз оттока «СвязьРегион». Ноутбук берёт таблицы четвёртого этапа (`features.csv`, `labels.csv`, `split.csv`), воспроизводит базовое правило на train и validation, обучает два метода в шести конфигурациях, выбирает кандидата по заранее записанному правилу и сохраняет его вместе с подготовкой признаков. В конце сохранённый кандидат загружается в новом процессе и без `fit` выдаёт те же validation-прогнозы.

**Обучение** — из корня репозитория, одной командой:

```
jupyter nbconvert --to notebook --execute --inplace lesson5/train.ipynb
```

**Проверка сохранённого кандидата** — отдельной командой, без обучения:

```
python lesson5/starter/check_saved_model.py --model lesson5/output/model/candidate.joblib --prepared lesson4/output --out lesson5/output/reload_validation_predictions.csv --expected lesson5/output/validation_predictions.csv
```

Ядро работает в папке `lesson5/`, все пути ниже относительны неё. Результаты — в `lesson5/output/`. Списки признаков, план экспериментов и правило выбора — `lesson5/config.json`.

| Параметр | Правило |
|---|---|
| Вход | `../lesson4/output/`: признаки, метки и разбиение четвёртого этапа, без изменений |
| Разбиение | train 01.12.2025–01.07.2026 (8 дат), validation 01.08.2026; только по времени, как на этапе 4 |
| Test 01.09.2026 | **не оценивается**: признаки и метки test не попадают ни в одну модель и ни в одну метрику |
| Лимит кампании | `K = floor(0,10 × N_active)` на каждой T; N — вся активная база; один K для baseline и моделей |
| Равные оценки | `customer_id` по возрастанию |
| Основная метрика | Recall@K; при нескольких датах — macro (среднее по датам), pooled показан рядом |
| Дополнительные | Precision@K и Average Precision (`sklearn.metrics.average_precision_score`, не трапецеидальная PR-AUC) |
| Обучение | каждый опыт — новый `Pipeline` (подготовка + классификатор), `fit` только на train |

Загрузка и расчёт метрик — функции из комплекта (`starter/training_utils.py`), подготовка для регрессии — `starter/preprocessing_template.py`. Файлы комплекта не изменяются.

---

## Шаг 0. Конфигурация

In [1]:
import hashlib
import json
import platform
import subprocess
import sys
import time
import warnings
from math import comb
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

if not Path("train.ipynb").exists():
    raise RuntimeError(f"Kernel must run in lesson5/, current folder is {Path.cwd()}")

STARTER = Path("starter")
# The saved Pipeline refers to preprocessing_template by module name, so it is imported
# from the same folder that check_saved_model.py runs in.
sys.path.insert(0, str(STARTER.resolve()))
from preprocessing_template import make_preprocessor
from training_utils import FORBIDDEN, KEY, campaign_metrics, load_train_validation, read_csv

RUN_CONFIG = Path("config.json")
OUTPUT = Path("output")
MODEL_DIR = OUTPUT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

VERSIONS = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "scikit-learn": sklearn.__version__, "joblib": joblib.__version__}
print("; ".join(f"{name} {version}" for name, version in VERSIONS.items()))

python 3.13.18; pandas 2.3.3; numpy 2.5.4; scikit-learn 1.9.1; joblib 1.6.0


In [2]:
CHECKS = []


def require(condition: bool, message: str) -> None:
    # Fail fast on a violated precondition.
    if not condition:
        raise ValueError(message)


def record_check(name: str, passed: bool, detail: str) -> None:
    # Checks are collected into output/checks.csv; a failed one stops the notebook in the last cell.
    CHECKS.append({"check": name, "passed": bool(passed), "detail": detail})
    print(f"[{'ok' if passed else 'FAILED'}] {name} — {detail}")


def write_csv(frame: pd.DataFrame, path: Path) -> None:
    frame.to_csv(path, sep=";", encoding="utf-8-sig", index=False, lineterminator="\n")


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def day(snapshot_at: pd.Series) -> pd.Series:
    # '2026-08-01T00:00:00+03:00' -> '2026-08-01'; every snapshot is midnight Moscow time.
    return snapshot_at.str.slice(0, 10)

Конфигурация читается из `config.json` и проверяется до загрузки данных. В ней два набора признаков:

- **`x_v1`** — все 24 признака этапа 4 (19 числовых, 5 категориальных);
- **`x_v2_no_digital`** — тот же набор без цифрового источника (минус 5 цифровых признаков и 2 индикатора его полноты, остаётся 17). Это обещанное в плане этапа 4 сравнение «с цифровой группой / без неё»: на train связь цифровых признаков с уходом не подтвердилась.

Списки явные. Ключи, метка, конец горизонта и разбиение в них попасть не могут: это проверяется здесь и ещё раз в загрузчике комплекта.

In [3]:
config = json.loads(RUN_CONFIG.read_text(encoding="utf-8"))
SEED = config["seed"]
FRACTION = config["campaign_fraction"]
PREPARED = Path(config["prepared"])
BASELINE_FEATURE = config["baseline_feature"]
FEATURE_SETS = config["feature_sets"]
EXPERIMENTS = config["experiments"]
FOLDS = config["temporal_folds"]
FAMILIES = {"logistic_regression": "Логистическая регрессия (L2)",
            "hist_gradient_boosting": "HistGradientBoostingClassifier"}

require(0 < FRACTION <= 0.10, "campaign_fraction must be in (0, 0.10]")
for name, feature_set in FEATURE_SETS.items():
    columns = feature_set["numeric_columns"] + feature_set["categorical_columns"]
    require(len(columns) == len(set(columns)), f"{name}: numeric and categorical lists overlap or repeat")
    require(not set(columns) & FORBIDDEN, f"{name}: keys, target or split in the feature list")
require(len({e["id"] for e in EXPERIMENTS}) == len(EXPERIMENTS), "experiment ids must be unique")
require(3 <= len(EXPERIMENTS) <= 8, "the assignment allows 3-6 configurations, 8 at most")
require({e["family"] for e in EXPERIMENTS} == set(FAMILIES), "both methods must be present, and only they")
require(all(e["feature_set"] in FEATURE_SETS for e in EXPERIMENTS), "experiment refers to an unknown feature set")

ALL_NUMERIC = FEATURE_SETS["x_v1"]["numeric_columns"]
ALL_CATEGORICAL = FEATURE_SETS["x_v1"]["categorical_columns"]
for name, feature_set in FEATURE_SETS.items():
    require(set(feature_set["numeric_columns"]) <= set(ALL_NUMERIC)
            and set(feature_set["categorical_columns"]) <= set(ALL_CATEGORICAL), f"{name} is not a subset of x_v1")
require(BASELINE_FEATURE in ALL_NUMERIC, "baseline feature must be a numeric column of the prepared table")

pd.DataFrame([{"набор": name, "числовых": len(fs["numeric_columns"]), "категориальных": len(fs["categorical_columns"]),
               "всего": len(fs["numeric_columns"]) + len(fs["categorical_columns"])} for name, fs in FEATURE_SETS.items()])

,набор,числовых,категориальных,всего
0,x_v1,19,5,24
1,x_v2_no_digital,13,4,17


*Шаг 0. Конфигурация*

---

## Шаг 1. Проверка входа

Три таблицы этапа 4 соединяются **по ключу** `(customer_id, snapshot_at)`, а не по порядку строк. Сначала проверяется сам вход, потом он загружается в модельный вид.

Версия данных — контрольные суммы трёх файлов. Они сравниваются с `lesson4/output/checksums.csv`: модель обучается ровно на тех таблицах, которые сдал этап 4.

In [4]:
INPUT_FILES = ["features.csv", "labels.csv", "split.csv"]
DATA_VERSION = {name: sha256(PREPARED / name) for name in INPUT_FILES}
lesson4_sums = read_csv(PREPARED / "checksums.csv").set_index("file").sha256
record_check("input files are byte-identical to the lesson 4 delivery",
             all(DATA_VERSION[name] == lesson4_sums[f"output/{name}"] for name in INPUT_FILES),
             "; ".join(f"{name} {DATA_VERSION[name][:12]}" for name in INPUT_FILES))

raw_features = read_csv(PREPARED / "features.csv")
raw_labels = read_csv(PREPARED / "labels.csv")
raw_split = read_csv(PREPARED / "split.csv")


def key_set(frame: pd.DataFrame) -> set:
    return set(map(tuple, frame[KEY].itertuples(index=False, name=None)))


record_check("key (customer_id, snapshot_at) is filled and unique in each table",
             all(not t[KEY].eq("").any().any() and not t.duplicated(KEY).any()
                 for t in (raw_features, raw_labels, raw_split)),
             f"features {len(raw_features)}, labels {len(raw_labels)}, split {len(raw_split)} rows")
record_check("the three tables hold the same key set",
             key_set(raw_features) == key_set(raw_labels) == key_set(raw_split),
             f"{len(key_set(raw_features))} keys in each")

split_dates = raw_split.assign(date=day(raw_split.snapshot_at)).groupby("split").date.agg(["min", "max", "nunique", "size"])
display(split_dates)
EXPECTED_DATES = {"train": ("2025-12-01", "2026-07-01", 8), "validation": ("2026-08-01", "2026-08-01", 1),
                  "test": ("2026-09-01", "2026-09-01", 1)}
record_check("split is the lesson 4 temporal split, not reshuffled",
             all(tuple(split_dates.loc[part, ["min", "max", "nunique"]]) == EXPECTED_DATES[part] for part in EXPECTED_DATES)
             and raw_split.groupby("snapshot_at").split.nunique().eq(1).all(),
             "train 2025-12-01..2026-07-01 (8 dates), validation 2026-08-01, test 2026-09-01; one part per date")

catalog = read_csv(Path("../lesson4/docs/feature_catalog.csv"))
catalog_x = set(catalog.feature_name[catalog.include_in_X == "yes"])
record_check("x_v1 is exactly the lesson 4 catalog list (include_in_X = yes)",
             set(ALL_NUMERIC + ALL_CATEGORICAL) == catalog_x and catalog_x <= set(raw_features.columns),
             f"{len(catalog_x)} features; keys, target, horizon end and audit fields are not among them")

[ok] input files are byte-identical to the lesson 4 delivery — features.csv 85d558c06d06; labels.csv 1054edfb6907; split.csv e68d40faaf60
[ok] key (customer_id, snapshot_at) is filled and unique in each table — features 19541, labels 19541, split 19541 rows
[ok] the three tables hold the same key set — 19541 keys in each


,min,max,nunique,size
split,,,,
test,2026-09-01,2026-09-01,1,1916
train,2025-12-01,2026-07-01,8,15696
validation,2026-08-01,2026-08-01,1,1929


[ok] split is the lesson 4 temporal split, not reshuffled — train 2025-12-01..2026-07-01 (8 dates), validation 2026-08-01, test 2026-09-01; one part per date
[ok] x_v1 is exactly the lesson 4 catalog list (include_in_X = yes) — 24 features; keys, target, horizon end and audit fields are not among them


Загрузка в модельный вид — `load_train_validation` из комплекта. Она ещё раз проверяет ключи, соединяет таблицы по ключу с `validate='one_to_one'`, переводит числа и категории в нужные типы и останавливается, если:

- метка train или validation неизвестна или не 0/1;
- в train нет одного из классов;
- горизонт метки какой-либо строки train заканчивается позже начала validation.

Строки test функция отбрасывает до чтения признаков: в `parts` их нет.

In [5]:
parts = load_train_validation(PREPARED, ALL_NUMERIC, ALL_CATEGORICAL)
for part in parts.values():
    part["date"] = day(part["keys"].snapshot_at).to_numpy()

n_test = int((raw_split.split == "test").sum())
record_check("no row is lost in the join",
             len(parts["train"]["y"]) + len(parts["validation"]["y"]) + n_test == len(raw_features),
             f"train {len(parts['train']['y'])} + validation {len(parts['validation']['y'])} + test {n_test} (not loaded) = {len(raw_features)}")

# Active base and churn counts per date against the independent recount of lesson 4.
base_check = read_csv(PREPARED / "check_active_base.csv").assign(date=lambda t: day(t.snapshot_at)).set_index("date")
loaded = pd.concat([pd.DataFrame({"date": p["date"], "y": p["y"]}) for p in parts.values()])
per_date = loaded.groupby("date").y.agg(n_active="size", n_churn="sum")
expected = base_check.loc[per_date.index, ["n_independent", "churn_independent"]].astype(int)
record_check("active base and churn count on every train/validation date equal the lesson 4 independent recount",
             (per_date.n_active == expected.n_independent).all() and (per_date.n_churn == expected.churn_independent).all(),
             f"{len(per_date)} dates; churn per date {per_date.n_churn.tolist()}")

train_ends = pd.to_datetime(parts["train"]["target_end"], utc=True)
validation_start = pd.to_datetime(parts["validation"]["keys"].snapshot_at, utc=True).min()
record_check("every train label is fully observed before validation starts",
             bool(train_ends.max() <= validation_start),
             f"last train horizon ends {train_ends.max():%Y-%m-%d %H:%M} UTC, validation T = {validation_start:%Y-%m-%d %H:%M} UTC")

X_train = parts["train"]["X"]
record_check("model input types: numeric columns are numbers, categorical are text",
             all(pd.api.types.is_numeric_dtype(X_train[c]) for c in ALL_NUMERIC)
             and all(X_train[c].dtype == object for c in ALL_CATEGORICAL)
             and list(X_train.columns) == ALL_NUMERIC + ALL_CATEGORICAL,
             f"{len(ALL_NUMERIC)} numeric + {len(ALL_CATEGORICAL)} text columns, order as in config.json")
record_check("both classes are present in train", set(parts["train"]["y"]) == {0, 1},
             f"{int(parts['train']['y'].sum())} churners of {len(parts['train']['y'])} rows "
             f"({parts['train']['y'].mean():.2%}); validation {int(parts['validation']['y'].sum())} of {len(parts['validation']['y'])}")
per_date.T

[ok] no row is lost in the join — train 15696 + validation 1929 + test 1916 (not loaded) = 19541
[ok] active base and churn count on every train/validation date equal the lesson 4 independent recount — 9 dates; churn per date [48, 50, 52, 54, 56, 58, 60, 62, 63]
[ok] every train label is fully observed before validation starts — last train horizon ends 2026-07-31 21:00 UTC, validation T = 2026-07-31 21:00 UTC
[ok] model input types: numeric columns are numbers, categorical are text — 19 numeric + 5 text columns, order as in config.json
[ok] both classes are present in train — 440 churners of 15696 rows (2.80%); validation 63 of 1929


date,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01,2026-06-01,2026-07-01,2026-08-01
n_active,1969,1971,1971,1969,1965,1959,1951,1941,1929
n_churn,48,50,52,54,56,58,60,62,63


*Шаг 1. Проверка входа*

---

## Шаг 2. Baseline на тех же данных

Правило этапа 4: на каждой T вся активная база сортируется по убыванию `debt_prev_rub` (долг на конец предыдущего месяца), равные значения — по `customer_id`. Неизвестный долг ставится после всех известных; это место в сортировке, а не восстановленное значение (на данных неизвестного долга нет).

Метрики считает та же функция комплекта `campaign_metrics`, что и для моделей: одинаковые база, K и правило равенств получаются по построению. Для baseline ей передаётся долг как score.

- `Recall@K = найдено / ушли`; если ушедших нет — неизвестно (пусто), не ноль.
- `Precision@K = найдено / K`; при K = 0 — неизвестно.
- Усреднение: **macro** — среднее долей по датам; **pooled** — сумма числителей / сумма знаменателей. На validation одна дата, там они совпадают.

In [6]:
def metric_rows(run_id: str, part_name: str, keys: pd.DataFrame, y: np.ndarray, score: np.ndarray):
    # Per-date table and one summary row; the ranking itself is done by the kit's campaign_metrics.
    _, by_date, ranked = campaign_metrics(keys, y, score, FRACTION)
    by_date = by_date.assign(snapshot_at=day(by_date.snapshot_at))
    by_date.insert(0, "part", part_name)
    by_date.insert(0, "id", run_id)
    summary = {
        "id": run_id, "part": part_name, "n_dates": len(by_date),
        "macro_recall_at_K": by_date.recall_at_K.mean(), "pooled_recall_at_K": by_date.hits.sum() / by_date.n_churn.sum(),
        "macro_precision_at_K": by_date.precision_at_K.mean(), "macro_average_precision": by_date.average_precision.mean(),
        "hits": int(by_date.hits.sum()), "n_churn": int(by_date.n_churn.sum()), "K_total": int(by_date.K.sum()),
    }
    return by_date, summary, ranked


def baseline_score(part: dict) -> np.ndarray:
    debt = part["X"][BASELINE_FEATURE]
    # Unknown debt ranks after every known value: a position, not an imputed amount.
    return debt.fillna(debt.min() - 1).to_numpy()


BY_DATE, SUMMARY, RANKED = [], [], {}
for part_name in ("train", "validation"):
    part = parts[part_name]
    by_date, summary, RANKED[("baseline", part_name)] = metric_rows("baseline", part_name, part["keys"], part["y"],
                                                                    baseline_score(part))
    BY_DATE.append(by_date)
    SUMMARY.append(summary)

baseline_by_date = pd.concat(BY_DATE, ignore_index=True)
unknown_debt = sum(int(parts[p]["X"][BASELINE_FEATURE].isna().sum()) for p in parts)
print(f"Строк с неизвестным долгом в train и validation: {unknown_debt}")
baseline_by_date.round(4)

Строк с неизвестным долгом в train и validation: 0


,id,part,snapshot_at,n_active,K,n_churn,hits,recall_at_K,precision_at_K,average_precision
0,baseline,train,2025-12-01,1969,196,48,16,0.3333,0.0816,0.0564
1,baseline,train,2026-01-01,1971,197,50,19,0.3800,0.0964,0.0945
2,baseline,train,2026-02-01,1971,197,52,21,0.4038,0.1066,0.0692
3,baseline,train,2026-03-01,1969,196,54,22,0.4074,0.1122,0.0971
4,baseline,train,2026-04-01,1965,196,56,23,0.4107,0.1173,0.0944
5,baseline,train,2026-05-01,1959,195,58,26,0.4483,0.1333,0.1019
6,baseline,train,2026-06-01,1951,195,60,23,0.3833,0.1179,0.1046
7,baseline,train,2026-07-01,1941,194,62,25,0.4032,0.1289,0.0878
8,baseline,validation,2026-08-01,1929,192,63,14,0.2222,0.0729,0.0583


Сверка с этапом 4: N, K и число найденных уходов на каждой дате train должны совпасть с `lesson4/output/baseline_train_metrics.csv`, посчитанным другим кодом.

In [7]:
lesson4_baseline = read_csv(PREPARED / "baseline_train_metrics.csv")
lesson4_baseline = lesson4_baseline[lesson4_baseline.row_type == "date"].set_index("snapshot_at")[["n_active", "k", "tp_at_k", "n_positive"]].astype(int)
mine = baseline_by_date[baseline_by_date.part == "train"].set_index("snapshot_at")[["n_active", "K", "hits", "n_churn"]]
record_check("baseline on train reproduces N, K, found and churned of lesson 4 on every date",
             bool((mine.to_numpy() == lesson4_baseline.loc[mine.index].to_numpy()).all()),
             f"found per date {mine.hits.tolist()}, K {mine.K.tolist()}")
pd.DataFrame(SUMMARY).round(4)

[ok] baseline on train reproduces N, K, found and churned of lesson 4 on every date — found per date [16, 19, 21, 22, 23, 26, 23, 25], K [196, 197, 197, 196, 196, 195, 195, 194]


,id,part,n_dates,macro_recall_at_K,pooled_recall_at_K,macro_precision_at_K,macro_average_precision,hits,n_churn,K_total
0,baseline,train,8,0.3963,0.3977,0.1118,0.0882,175,440,1566
1,baseline,validation,1,0.2222,0.2222,0.0729,0.0583,14,63,192


На train правило находит 175 из 440 уходов: macro Recall@K 39,6%, pooled 39,8% — те же числа, что на этапе 4. На validation 01.08.2026 — **14 из 63, Recall@K 22,2%**, Precision@K 7,3%.

Правило ничего не обучает, поэтому падение с 39,6% до 22,2% — не переобучение, а свойство августа: на датах train оно находило от 33,3% до 44,8%. Отсюда два следствия для остального ноутбука. Модели на validation нужно сравнивать с 22,2%, а не с train-уровнем правила. И разрыв train/validation у моделей нельзя целиком считать переобучением: часть его — тот же эффект месяца.

*Шаг 2. Baseline на тех же данных*

---

## Шаг 3. Два обучаемых метода

Методы — те, что выбраны в плане этапа 4; замены нет.

| | Кандидат 1: логистическая регрессия (L2) | Кандидат 2: `HistGradientBoostingClassifier` |
|---|---|---|
| Числа | медиана train + флаг пропуска, стандартизация | без изменений: пропуск обрабатывается деревом как отдельное направление |
| Категории | пропуск → `__MISSING__`, One-Hot; новая категория → нулевой вектор | порядковые коды, обученные на train, и `categorical_features`; новая категория и пропуск → «пропуск» |
| Масштабирование | нужно (регуляризация зависит от масштаба) | не нужно |
| Остановка | `lbfgs`, до сходимости, `max_iter=2000` | фиксированное число деревьев, `early_stopping=False` |
| Подготовка | `make_preprocessor` из комплекта | собственный `ColumnTransformer` (ниже) |

Для бустинга подготовка отличается от примера комплекта (там общий One-Hot с медианой): план этапа 4 записал нативную работу с пропусками и категориями, и так модель видит пропуск как пропуск, а не как медиану. Автоматическая ранняя остановка выключена: у sklearn она берёт случайную внутреннюю подвыборку, что смешало бы даты. Число деревьев задано заранее в плане экспериментов.

Подготовка и классификатор собраны в один `Pipeline`. `fit` вызывается у всего Pipeline и получает только строки train, поэтому медианы, масштабы и коды категорий физически не могут зависеть от validation. Веса классов и ресемплинг не используются: метрика ранжирующая, а сдвиг вероятностей от балансировки порядок почти не меняет.

In [8]:
def columns_of(experiment: dict) -> tuple[list, list]:
    feature_set = FEATURE_SETS[experiment["feature_set"]]
    return feature_set["numeric_columns"], feature_set["categorical_columns"]


def build_pipeline(experiment: dict) -> Pipeline:
    # A new, unfitted Pipeline (preprocessing + classifier) for one experiment.
    numeric, categorical = columns_of(experiment)
    params = dict(experiment["params"])
    if experiment["family"] == "logistic_regression":
        return Pipeline([("preprocess", make_preprocessor(numeric, categorical)),
                         ("model", LogisticRegression(solver="lbfgs", max_iter=2000, random_state=SEED, **params))])
    if experiment["family"] == "hist_gradient_boosting":
        require("early_stopping" not in params, "early stopping needs a temporal inner split, a random one is not allowed")
        encode = ColumnTransformer([
            ("numeric", "passthrough", numeric),
            ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), categorical),
        ], remainder="drop")
        categorical_positions = list(range(len(numeric), len(numeric) + len(categorical)))
        return Pipeline([("preprocess", encode),
                         ("model", HistGradientBoostingClassifier(early_stopping=False, random_state=SEED,
                                                                  categorical_features=categorical_positions, **params))])
    raise ValueError(f"Unknown family: {experiment['family']}")


def churn_score(pipeline: Pipeline, X: pd.DataFrame) -> np.ndarray:
    # Continuous score = predict_proba column of class 1, located by classes_, not by position.
    classes = list(pipeline.classes_)
    require(classes == [0, 1], f"unexpected classes {classes}")
    return pipeline.predict_proba(X)[:, classes.index(1)]


def run(experiment: dict, fit_part: dict, score_parts: dict):
    # Fit a fresh Pipeline on fit_part only, then score the given parts with the fitted Pipeline.
    columns = [*columns_of(experiment)[0], *columns_of(experiment)[1]]
    pipeline = build_pipeline(experiment)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        started = time.perf_counter()
        pipeline.fit(fit_part["X"][columns], fit_part["y"])
        fit_seconds = time.perf_counter() - started
    scores = {name: churn_score(pipeline, part["X"][columns]) for name, part in score_parts.items()}
    return pipeline, scores, fit_seconds, [f"{w.category.__name__}: {w.message}" for w in caught]


def rows_of(part: dict, mask: np.ndarray) -> dict:
    return {"X": part["X"][mask].reset_index(drop=True), "y": part["y"][mask],
            "keys": part["keys"][mask].reset_index(drop=True), "date": part["date"][mask]}

*Шаг 3. Два обучаемых метода*

---

## Шаг 4. План экспериментов и запуск

План и правило выбора записаны в `config.json` **до** запуска. Шесть конфигураций, по три на метод:

| ID | Метод | Набор X | Что меняется | Зачем |
|---|---|---|---|---|
| `lr_c01` | Регрессия | `x_v1` | `C = 0,1` — сильная регуляризация | признаки долга коррелируют (ρ = 0,79); сжатие коэффициентов может быть устойчивее |
| `lr_c1` | Регрессия | `x_v1` | `C = 1` — слабая | проверка, что регуляризация вообще что-то меняет при 15 696 строках |
| `hgb_leaf4` | Бустинг | `x_v1` | 4 листа на дерево | почти аддитивная модель: пороги по долгу, минимум взаимодействий |
| `hgb_leaf8` | Бустинг | `x_v1` | 8 листьев | допускает взаимодействия (стаж × долг); выше риск переобучения при 440 уходах |
| `lr_c1_nodig` | Регрессия | `x_v2_no_digital` | убран цифровой источник | вносит ли цифровая группа сигнал или шум |
| `hgb_leaf4_nodig` | Бустинг | `x_v2_no_digital` | убран цифровой источник | то же для бустинга |

Остальные параметры бустинга одинаковы во всех опытах: 100 деревьев, `learning_rate = 0,05`, `min_samples_leaf = 100`, `l2_regularization = 1`. Seed один — 42; регрессия с `lbfgs` и бустинг без подвыборки от seed не зависят, разные seed как разные опыты не подаются.

**Правило выбора** (в порядке применения):

1. максимальный validation Recall@K;
2. при равенстве — максимальный pooled Recall@K на временных фолдах внутри train;
3. при равенстве — максимальный validation Average Precision;
4. при полном равенстве — первый по порядку в `config.json`.

**Временные фолды** — дополнительная проверка устойчивости из плана этапа 4. Каждая конфигурация обучается на датах до границы и проверяется на следующей дате: обучение по 01.03 → проверка 01.04.2026, затем по 01.04 → 01.05, по 01.05 → 01.06, по 01.06 → 01.07. Весь Pipeline, включая подготовку, обучается внутри фолда заново. Метка последней обучающей даты заканчивается ровно к проверочной T. На четырёх проверочных датах 236 уходов против 63 на validation, поэтому фолды стоят вторым шагом правила: при одинаковом числе найденных на validation решает более крупная выборка.

> **Отступление от плана этапа 4.** Там параметры подбирались только по фолдам, а validation использовалась один раз. Задание этапа 5 требует train- и validation-метрик для каждого опыта и выбора по validation, поэтому основной критерий — validation, фолды — второй. Последствие названо прямо: validation-качество выбранного кандидата после выбора из шести вариантов — оптимистичная оценка.

In [9]:
for fold in FOLDS:
    require(fold["train_end_inclusive"] < fold["check_date"] and fold["check_date"] in set(parts["train"]["date"]),
            f"fold {fold} must check on a later train date")

LOG, FOLD_BY_DATE, PIPELINES, SCORES = [], [], {}, {}
train, validation = parts["train"], parts["validation"]

for order, experiment in enumerate(EXPERIMENTS, start=1):
    run_id = experiment["id"]
    pipeline, scores, fit_seconds, caught = run(experiment, train, {"train": train, "validation": validation})
    PIPELINES[run_id], SCORES[run_id] = pipeline, scores
    for part_name in ("train", "validation"):
        part = parts[part_name]
        by_date, summary, RANKED[(run_id, part_name)] = metric_rows(run_id, part_name, part["keys"], part["y"], scores[part_name])
        BY_DATE.append(by_date)
        SUMMARY.append(summary)

    # Temporal folds inside train: the whole Pipeline is refitted on the earlier dates only.
    for fold in FOLDS:
        fit_rows = rows_of(train, train["date"] <= fold["train_end_inclusive"])
        check_rows = rows_of(train, train["date"] == fold["check_date"])
        _, fold_scores, _, fold_caught = run(experiment, fit_rows, {"check": check_rows})
        caught += fold_caught
        by_date, _, _ = metric_rows(run_id, "fold", check_rows["keys"], check_rows["y"], fold_scores["check"])
        by_date.insert(2, "fit_through", fold["train_end_inclusive"])
        FOLD_BY_DATE.append(by_date)

    numeric, categorical = columns_of(experiment)
    LOG.append({"order": order, "id": run_id, "method": FAMILIES[experiment["family"]],
                "params": json.dumps(experiment["params"]), "seed": SEED, "x_version": experiment["feature_set"],
                "n_features": len(numeric) + len(categorical), "n_model_columns": pipeline[:-1].transform(train["X"][numeric + categorical][:1]).shape[1],
                "fit_seconds": round(fit_seconds, 2),
                "status": "ok" if not caught else "ok, warnings", "warnings": " | ".join(sorted(set(caught)))})
    print(f"{run_id}: fit {fit_seconds:.2f} s, warnings: {len(caught)}")

log = pd.DataFrame(LOG)
by_date_all = pd.concat(BY_DATE, ignore_index=True)
fold_by_date = pd.concat(FOLD_BY_DATE, ignore_index=True)
summary_all = pd.DataFrame(SUMMARY)
log.drop(columns="warnings")

lr_c01: fit 0.11 s, warnings: 0


lr_c1: fit 0.10 s, warnings: 0


hgb_leaf4: fit 0.15 s, warnings: 0


hgb_leaf8: fit 0.20 s, warnings: 0


lr_c1_nodig: fit 0.06 s, warnings: 0


hgb_leaf4_nodig: fit 0.13 s, warnings: 0


,order,id,method,params,seed,x_version,n_features,n_model_columns,fit_seconds,status
0,1,lr_c01,Логистическая регрессия (L2),"{""C"": 0.1}",42,x_v1,24,45,0.11,ok
1,2,lr_c1,Логистическая регрессия (L2),"{""C"": 1.0}",42,x_v1,24,45,0.10,ok
2,3,hgb_leaf4,HistGradientBoostingClassifier,"{""max_leaf_nodes"": 4, ""max_iter"": 100, ""learning_rate"": 0.05, ""min_samples_leaf"": 100, ""l2_regularization"": 1.0}",42,x_v1,24,24,0.15,ok
3,4,hgb_leaf8,HistGradientBoostingClassifier,"{""max_leaf_nodes"": 8, ""max_iter"": 100, ""learning_rate"": 0.05, ""min_samples_leaf"": 100, ""l2_regularization"": 1.0}",42,x_v1,24,24,0.20,ok
4,5,lr_c1_nodig,Логистическая регрессия (L2),"{""C"": 1.0}",42,x_v2_no_digital,17,30,0.06,ok
5,6,hgb_leaf4_nodig,HistGradientBoostingClassifier,"{""max_leaf_nodes"": 4, ""max_iter"": 100, ""learning_rate"": 0.05, ""min_samples_leaf"": 100, ""l2_regularization"": 1.0}",42,x_v2_no_digital,17,17,0.13,ok


### Проверки обучения

- **Fit только на train.** Медианы подготовки регрессии равны медианам train, посчитанным напрямую; коды категорий бустинга — категории train.
- **Новая категория и пропуск категории** в validation обрабатываются без ошибки и без `fit`; оценки остальных строк не меняются.
- **Порядок строк.** Перевёрнутая validation даёт те же оценки в перевёрнутом порядке: оценка строки зависит только от её признаков.
- **Сходимость.** Предупреждения перехватываются при каждом `fit` и попадают в журнал; пустой столбец означает, что их не было.

In [10]:
lr_id = next(e["id"] for e in EXPERIMENTS if e["family"] == "logistic_regression" and e["feature_set"] == "x_v1")
hgb_id = next(e["id"] for e in EXPERIMENTS if e["family"] == "hist_gradient_boosting" and e["feature_set"] == "x_v1")
columns_v1 = ALL_NUMERIC + ALL_CATEGORICAL

medians = PIPELINES[lr_id].named_steps["preprocess"].named_transformers_["numeric"].named_steps["impute"].statistics_
hgb_categories = PIPELINES[hgb_id].named_steps["preprocess"].named_transformers_["categorical"].categories_
record_check("preprocessing statistics come from train only",
             bool(np.allclose(medians, train["X"][ALL_NUMERIC].median().to_numpy()))
             and all(set(c) == set(train["X"][col].dropna()) for c, col in zip(hgb_categories, ALL_CATEGORICAL)),
             f"{len(medians)} medians equal train medians; boosting category codes equal train categories")

novel = validation["X"][columns_v1].copy()
novel.loc[0, "tariff_prev"] = "Ultra"
novel.loc[1, "region"] = "Новая область"
novel.loc[2, "tariff_prev"] = np.nan
novel.loc[3, "digital_status_prev"] = "pending_migration"
novel_ok = True
for run_id in (lr_id, hgb_id):
    novel_scores = churn_score(PIPELINES[run_id], novel)
    novel_ok &= bool(np.isfinite(novel_scores).all() and np.array_equal(novel_scores[4:], SCORES[run_id]["validation"][4:]))
record_check("unseen and missing categories in validation are scored without refit", novel_ok,
             "4 changed rows get finite scores, the other rows keep their scores exactly (both methods)")

reversed_X = validation["X"][columns_v1].iloc[::-1].reset_index(drop=True)
# Tolerance instead of exact equality: BLAS may sum a reordered matrix in a different order.
record_check("a row's score does not depend on row order",
             all(np.allclose(churn_score(PIPELINES[i], reversed_X)[::-1], SCORES[i]["validation"], rtol=0, atol=1e-12)
                 for i in (lr_id, hgb_id)), "reversed validation gives the reversed scores (both methods)")

record_check("every experiment trained and converged without warnings", bool((log.status == "ok").all()),
             f"{len(log)} experiments, {len(FOLDS)} temporal folds each; warnings: {log.warnings[log.warnings != ''].tolist() or 'none'}")
record_check("test rows were not loaded, fitted on or scored",
             all(set(p["date"]) <= {d for d in per_date.index} for p in parts.values()) and "2026-09-01" not in per_date.index,
             f"dates in memory: {sorted(per_date.index)[0]}..{sorted(per_date.index)[-1]}")

[ok] preprocessing statistics come from train only — 19 medians equal train medians; boosting category codes equal train categories
[ok] unseen and missing categories in validation are scored without refit — 4 changed rows get finite scores, the other rows keep their scores exactly (both methods)
[ok] a row's score does not depend on row order — reversed validation gives the reversed scores (both methods)
[ok] every experiment trained and converged without warnings — 6 experiments, 4 temporal folds each; warnings: none
[ok] test rows were not loaded, fitted on or scored — dates in memory: 2025-12-01..2026-08-01


*Шаг 4. План экспериментов и запуск*

---

## Шаг 5. Оценки риска и метрики кампании

Score каждой модели — столбец `predict_proba` класса 1, найденный по `classes_`. На каждой T вся активная база сортируется по score по убыванию, равные — по `customer_id`; берутся первые K. Score не округляется, порог 0,5 не используется: в список попадают K абонентов с наибольшим риском, каким бы малым он ни был в абсолютном выражении.

Сводная таблица: train и validation рядом, плюс фолды. `gap` — разность train и validation Recall@K: чем она больше, тем сильнее модель подстроилась под обучающие строки.

In [11]:
def wide(metric: str) -> pd.DataFrame:
    return summary_all.pivot(index="id", columns="part", values=metric)


fold_summary = fold_by_date.groupby("id").agg(fold_hits=("hits", "sum"), fold_churn=("n_churn", "sum"),
                                              fold_macro_recall=("recall_at_K", "mean"),
                                              fold_min_recall=("recall_at_K", "min"), fold_max_recall=("recall_at_K", "max"))
fold_summary["fold_pooled_recall"] = fold_summary.fold_hits / fold_summary.fold_churn

results = pd.DataFrame({
    "train_recall": wide("macro_recall_at_K")["train"], "train_pooled_recall": wide("pooled_recall_at_K")["train"],
    "validation_recall": wide("macro_recall_at_K")["validation"],
    "validation_hits": wide("hits")["validation"], "validation_churn": wide("n_churn")["validation"],
    "validation_precision": wide("macro_precision_at_K")["validation"],
    "train_ap": wide("macro_average_precision")["train"], "validation_ap": wide("macro_average_precision")["validation"],
}).join(fold_summary[["fold_pooled_recall", "fold_hits", "fold_churn", "fold_min_recall", "fold_max_recall"]])
results["gap_train_validation"] = results.train_recall - results.validation_recall
results = results.loc[["baseline", *log.id]].reset_index()
results.round(4)

,id,train_recall,train_pooled_recall,validation_recall,validation_hits,validation_churn,validation_precision,train_ap,validation_ap,fold_pooled_recall,fold_hits,fold_churn,fold_min_recall,fold_max_recall,gap_train_validation
0,baseline,0.3963,0.3977,0.2222,14,63,0.0729,0.0882,0.0583,NaN,NaN,NaN,NaN,NaN,0.1740
1,lr_c01,0.4035,0.4023,0.2540,16,63,0.0833,0.1106,0.0631,0.3644,86.0,236.0,0.3387,0.3966,0.1495
2,lr_c1,0.4034,0.4023,0.2540,16,63,0.0833,0.1118,0.0638,0.3729,88.0,236.0,0.3387,0.4138,0.1494
3,hgb_leaf4,0.4573,0.4591,0.2698,17,63,0.0885,0.1578,0.0728,0.4110,97.0,236.0,0.3621,0.4516,0.1875
4,hgb_leaf8,0.5574,0.5591,0.2540,16,63,0.0833,0.2698,0.0711,0.3856,91.0,236.0,0.3103,0.4500,0.3034
5,lr_c1_nodig,0.3963,0.3955,0.2540,16,63,0.0833,0.1086,0.0643,0.3729,88.0,236.0,0.3500,0.3966,0.1423
6,hgb_leaf4_nodig,0.4561,0.4568,0.2857,18,63,0.0938,0.1483,0.0702,0.3898,92.0,236.0,0.3448,0.4194,0.1704


Для baseline фолды не считаются отдельно: правило ничего не обучает, его результат на проверочных датах фолдов — это его же train-метрики на 01.04–01.07.2026. Они подставляются ниже, чтобы сравнивать модели и правило на одних и тех же четырёх датах.

In [12]:
fold_dates = [fold["check_date"] for fold in FOLDS]
baseline_on_fold_dates = baseline_by_date[(baseline_by_date.part == "train") & baseline_by_date.snapshot_at.isin(fold_dates)]
is_baseline = results.id == "baseline"
results.loc[is_baseline, ["fold_hits", "fold_churn"]] = [baseline_on_fold_dates.hits.sum(), baseline_on_fold_dates.n_churn.sum()]
results.loc[is_baseline, "fold_pooled_recall"] = baseline_on_fold_dates.hits.sum() / baseline_on_fold_dates.n_churn.sum()
results.loc[is_baseline, ["fold_min_recall", "fold_max_recall"]] = [baseline_on_fold_dates.recall_at_K.min(), baseline_on_fold_dates.recall_at_K.max()]
results[["fold_hits", "fold_churn", "validation_hits", "validation_churn"]] = results[["fold_hits", "fold_churn", "validation_hits", "validation_churn"]].astype(int)

fold_table = (pd.concat([baseline_on_fold_dates.assign(id="baseline"), fold_by_date])
              .pivot(index="id", columns="snapshot_at", values="hits").loc[["baseline", *log.id]])
fold_table["всего"] = fold_table.sum(axis=1)
print("Найдено уходов на проверочных датах фолдов (ушли: "
      + ", ".join(f"{d} — {n}" for d, n in baseline_on_fold_dates.set_index("snapshot_at").n_churn.items()) + ")")
display(fold_table)
results[["id", "train_recall", "fold_pooled_recall", "validation_recall", "validation_hits", "validation_precision",
         "validation_ap", "gap_train_validation"]].round(4)

Найдено уходов на проверочных датах фолдов (ушли: 2026-04-01 — 56, 2026-05-01 — 58, 2026-06-01 — 60, 2026-07-01 — 62)


snapshot_at,2026-04-01,2026-05-01,2026-06-01,2026-07-01,всего
id,,,,,
baseline,23,26,23,25,97
lr_c01,19,23,23,21,86
lr_c1,20,24,23,21,88
hgb_leaf4,22,21,26,28,97
hgb_leaf8,20,18,27,26,91
lr_c1_nodig,20,23,21,24,88
hgb_leaf4_nodig,21,20,25,26,92


,id,train_recall,fold_pooled_recall,validation_recall,validation_hits,validation_precision,validation_ap,gap_train_validation
0,baseline,0.3963,0.4110,0.2222,14,0.0729,0.0583,0.1740
1,lr_c01,0.4035,0.3644,0.2540,16,0.0833,0.0631,0.1495
2,lr_c1,0.4034,0.3729,0.2540,16,0.0833,0.0638,0.1494
3,hgb_leaf4,0.4573,0.4110,0.2698,17,0.0885,0.0728,0.1875
4,hgb_leaf8,0.5574,0.3856,0.2540,16,0.0833,0.0711,0.3034
5,lr_c1_nodig,0.3963,0.3729,0.2540,16,0.0833,0.0643,0.1423
6,hgb_leaf4_nodig,0.4561,0.3898,0.2857,18,0.0938,0.0702,0.1704


Таблица по каждой T: числители и знаменатели для baseline и всех опытов на train и validation (`output/metrics_by_date.csv`). Ниже — validation целиком и найденные уходы по датам train.

In [13]:
display(by_date_all[by_date_all.part == "validation"].drop(columns="part").round(4))
train_hits = by_date_all[by_date_all.part == "train"].pivot(index="id", columns="snapshot_at", values="hits").loc[["baseline", *log.id]]
train_hits["всего"] = train_hits.sum(axis=1)
print("Найдено уходов на датах train (модели видели эти метки при обучении)")
train_hits

,id,snapshot_at,n_active,K,n_churn,hits,recall_at_K,precision_at_K,average_precision
8,baseline,2026-08-01,1929,192,63,14,0.2222,0.0729,0.0583
17,lr_c01,2026-08-01,1929,192,63,16,0.2540,0.0833,0.0631
26,lr_c1,2026-08-01,1929,192,63,16,0.2540,0.0833,0.0638
35,hgb_leaf4,2026-08-01,1929,192,63,17,0.2698,0.0885,0.0728
44,hgb_leaf8,2026-08-01,1929,192,63,16,0.2540,0.0833,0.0711
53,lr_c1_nodig,2026-08-01,1929,192,63,16,0.2540,0.0833,0.0643
62,hgb_leaf4_nodig,2026-08-01,1929,192,63,18,0.2857,0.0938,0.0702


Найдено уходов на датах train (модели видели эти метки при обучении)


snapshot_at,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01,2026-06-01,2026-07-01,всего
id,,,,,,,,,
baseline,16,19,21,22,23,26,23,25,175
lr_c01,20,23,20,20,23,25,23,23,177
lr_c1,20,23,20,20,22,26,23,23,177
hgb_leaf4,20,25,21,23,25,28,28,32,202
hgb_leaf8,24,29,25,31,33,36,33,35,246
lr_c1_nodig,20,22,19,20,22,25,22,24,174
hgb_leaf4_nodig,20,26,21,24,26,27,27,30,201


In [14]:
# Ties: with a continuous score the customer_id rule should rarely decide who enters the list.
tie_rows = []
for run_id in ["baseline", *log.id]:
    ranked = RANKED[(run_id, "validation")]
    cutoff = ranked.loc[ranked["rank"] == ranked.selected.sum(), "score"].iloc[0]
    tie_rows.append({"id": run_id, "K": int(ranked.selected.sum()), "score_at_cutoff": cutoff,
                     "n_equal_to_cutoff": int((ranked.score == cutoff).sum()),
                     "n_distinct_scores": int(ranked.score.nunique())})
ties = pd.DataFrame(tie_rows)
record_check("baseline and every model are ranked on the same base with the same K on each date",
             by_date_all.groupby(["part", "snapshot_at"])[["n_active", "K", "n_churn"]].nunique().eq(1).all().all()
             and fold_by_date.groupby("snapshot_at")[["n_active", "K", "n_churn"]].nunique().eq(1).all().all(),
             f"{by_date_all.id.nunique()} approaches x {by_date_all.snapshot_at.nunique()} dates; K = floor({FRACTION} x N)")
ties

[ok] baseline and every model are ranked on the same base with the same K on each date — 7 approaches x 9 dates; K = floor(0.1 x N)


,id,K,score_at_cutoff,n_equal_to_cutoff,n_distinct_scores
0,baseline,192,568.670000,1,336
1,lr_c01,192,0.069795,1,1929
2,lr_c1,192,0.068948,1,1929
3,hgb_leaf4,192,0.074828,1,1676
4,hgb_leaf8,192,0.078470,1,1899
5,lr_c1_nodig,192,0.069785,1,1929
6,hgb_leaf4_nodig,192,0.077491,1,1365


**Что видно в таблицах.**

- **Validation.** Все шесть моделей нашли больше правила: 16–18 уходов против 14 из 63. Разброс между моделями — два ухода; один уход = 1,6 п.п. Recall@K.
- **Регуляризация регрессии** ничего не меняет: `C = 0,1` и `C = 1` дают 16 и 16 на validation, 86 и 88 на фолдах, train-метрики совпадают до третьего знака. При 15 696 строках и 45 столбцах штраф почти не действует.
- **Сложность бустинга — переобучение.** От 4 к 8 листьям train Recall@K растёт с 45,7% до 55,7% (202 → 246 найденных), а на невиденных датах падает: фолды 97 → 91, validation 17 → 16. Рост только на train — признак запоминания, `hgb_leaf8` хуже.
- **Разрыв train/validation.** У правила он 17,4 п.п. без всякого обучения. У регрессии 14–15 п.п. — не больше, чем у правила, то есть это эффект месяца. У `hgb_leaf4` 18,8 п.п., у `hgb_leaf8` 30,3 п.п. — здесь около 13 п.п. сверх эффекта месяца приходится на переобучение.
- **Цифровой источник** устойчивого вклада не показал: у регрессии с ним и без него 16 и 16 (фолды 88 и 88), у бустинга 17 и 18 на validation, но 97 и 92 на фолдах — знаки противоположны.
- **Фолды против validation.** На четырёх проверочных датах внутри train ни одна модель не обошла правило: у правила 97 из 236, лучшая модель (`hgb_leaf4`) — тоже 97, остальные 86–92. Преимущество моделей видно только на августе.
- **Равенства.** На границе K ни у одного подхода нет равных оценок, правило `customer_id` на состав списков validation не повлияло. У бустинга 1 365–1 899 разных значений на 1 929 абонентов: внутри списка равные оценки есть, на границе — нет.

Train-метрики моделей — оценка на строках, метки которых модель видела. Они показаны для разрыва, а не как ожидаемое качество.

*Шаг 5. Оценки риска и метрики кампании*

---

## Шаг 6. Выбор и фиксация кандидата

Правило из шага 4 применяется механически: сортировка опытов по четырём ключам. Baseline в выборе не участвует — он не обучаемый метод, с ним кандидат сравнивается после выбора.

In [15]:
models = results[results.id != "baseline"].merge(log[["id", "order", "method", "x_version"]], on="id")
ranking = models.sort_values(["validation_recall", "fold_pooled_recall", "validation_ap", "order"],
                             ascending=[False, False, False, True]).reset_index(drop=True)
ranking.insert(0, "place", ranking.index + 1)
SELECTED = ranking.id.iloc[0]
selected_experiment = next(e for e in EXPERIMENTS if e["id"] == SELECTED)
decided_by = ("validation Recall@K" if ranking.validation_recall.iloc[0] > ranking.validation_recall.iloc[1]
              else "fold pooled Recall@K" if ranking.fold_pooled_recall.iloc[0] > ranking.fold_pooled_recall.iloc[1]
              else "validation AP" if ranking.validation_ap.iloc[0] > ranking.validation_ap.iloc[1] else "config order")
print(f"Выбран: {SELECTED}; решающий шаг правила: {decided_by}")
ranking[["place", "id", "method", "x_version", "validation_recall", "validation_hits", "fold_pooled_recall", "validation_ap", "order"]].round(4)

Выбран: hgb_leaf4_nodig; решающий шаг правила: validation Recall@K


,place,id,method,x_version,validation_recall,validation_hits,fold_pooled_recall,validation_ap,order
0,1,hgb_leaf4_nodig,HistGradientBoostingClassifier,x_v2_no_digital,0.2857,18,0.3898,0.0702,6
1,2,hgb_leaf4,HistGradientBoostingClassifier,x_v1,0.2698,17,0.4110,0.0728,3
2,3,hgb_leaf8,HistGradientBoostingClassifier,x_v1,0.2540,16,0.3856,0.0711,4
3,4,lr_c1_nodig,Логистическая регрессия (L2),x_v2_no_digital,0.2540,16,0.3729,0.0643,5
4,5,lr_c1,Логистическая регрессия (L2),x_v1,0.2540,16,0.3729,0.0638,2
5,6,lr_c01,Логистическая регрессия (L2),x_v1,0.2540,16,0.3644,0.0631,1


### Сравнение с baseline на той же validation

Списки кандидата и правила сравниваются поимённо: сколько абонентов общих и сколько ушедших нашёл только один из подходов. Перевес проверяется точным знаковым тестом по несовпадающим находкам: если подходы равноценны, каждый «спорный» ушедший с равной вероятностью достаётся любому из них.

In [16]:
def selected_set(run_id: str, part_name: str = "validation") -> pd.DataFrame:
    ranked = RANKED[(run_id, part_name)]
    return ranked[ranked.selected]


candidate_top, baseline_top = selected_set(SELECTED), selected_set("baseline")
candidate_found = set(candidate_top.customer_id[candidate_top.target == 1])
baseline_found = set(baseline_top.customer_id[baseline_top.target == 1])
only_candidate, only_baseline = len(candidate_found - baseline_found), len(baseline_found - candidate_found)
discordant = only_candidate + only_baseline
# Exact two-sided sign test on the discordant churners.
p_value = min(1.0, 2 * sum(comb(discordant, i) for i in range(min(only_candidate, only_baseline) + 1)) / 2 ** discordant) if discordant else 1.0

row = results.set_index("id")
comparison = pd.DataFrame([{
    "K": len(candidate_top), "ушли": int(row.validation_churn[SELECTED]),
    "нашёл baseline": len(baseline_found), "нашёл кандидат": len(candidate_found),
    "дополнительно найдено": len(candidate_found) - len(baseline_found),
    "общих абонентов в списках": len(set(candidate_top.customer_id) & set(baseline_top.customer_id)),
    "ушедших нашли оба": len(candidate_found & baseline_found),
    "только кандидат": only_candidate, "только baseline": only_baseline, "p (знаковый тест)": round(p_value, 3),
}])
fold_gain = int(row.fold_hits[SELECTED] - row.fold_hits["baseline"])
print(f"Временные фолды (4 даты, {int(row.fold_churn[SELECTED])} уходов): кандидат {int(row.fold_hits[SELECTED])}, "
      f"baseline {int(row.fold_hits['baseline'])}, разница {fold_gain:+d}")
unseen_candidate = int(row.fold_hits[SELECTED]) + len(candidate_found)
unseen_baseline = int(row.fold_hits["baseline"]) + len(baseline_found)
print(f"Все 5 дат, не виденных моделью при обучении ({int(row.fold_churn[SELECTED]) + int(row.validation_churn[SELECTED])} уходов): "
      f"кандидат {unseen_candidate}, baseline {unseen_baseline}")
comparison.T.rename(columns={0: "validation 01.08.2026"})

Временные фолды (4 даты, 236 уходов): кандидат 92, baseline 97, разница -5
Все 5 дат, не виденных моделью при обучении (299 уходов): кандидат 110, baseline 111


,validation 01.08.2026
K,192.000
ушли,63.000
нашёл baseline,14.000
нашёл кандидат,18.000
дополнительно найдено,4.000
общих абонентов в списках,162.000
ушедших нашли оба,14.000
только кандидат,4.000
только baseline,0.000
p (знаковый тест),0.125


**Выбран `hgb_leaf4_nodig`** — бустинг с 4 листьями на наборе без цифрового источника — по первому шагу правила: 18 найденных на validation, у следующего 17. Правило записано до запуска и применено без поправок.

**Сравнение с baseline на validation 01.08.2026:** 18 против 14 из 63, Recall@K 28,6% против 22,2%, **+4 найденных ухода** при том же K = 192. Списки совпадают на 162 абонента из 192. Все 14 ушедших, найденных правилом, нашёл и кандидат; ещё 4 нашёл только он.

**Насколько этому можно верить.** Слабо:

- 4 несовпадающих находки из 4 в одну сторону — p = 0,125 по знаковому тесту. С такой выборкой перевес от случайности не отличить.
- На четырёх проверочных датах внутри train та же конфигурация нашла 92 ухода против 97 у правила (−5). По всем пяти датам, которые модель при обучении не видела, — 110 против 111 из 299: паритет.
- Кандидат выбран как лучший из шести по этому же месяцу, а первое и второе место разделяет один уход. Его 28,6% — оптимистичная оценка.
- Август — нетипичный месяц для правила долга (22,2% против 33–45% на train). Возможно, модель выигрывает именно там, где долг работает плохо; одной датой это не проверить.

**Вывод.** Кандидат зафиксирован для шестого этапа, но преимущество над сортировкой по долгу **не доказано**. Модель не объявляется готовой к внедрению; правило долга остаётся рабочей альтернативой. Фолды здесь считались на переобученных копиях той же конфигурации с более коротким train, а не на сохранённом объекте: они говорят об устойчивости конфигурации, а не о качестве сохранённого Pipeline.

Test не оценивался. Обучения на train + validation не было.

*Шаг 6. Выбор и фиксация кандидата*

---

## Шаг 7. Сохранение и воспроизводимость

Сохраняется **тот самый** обученный на train Pipeline, который участвовал в сравнении: подготовка и классификатор одним объектом. Повторного обучения на train + validation нет.

- `output/model/candidate.joblib` — bundle в структуре стартового ноутбука: `pipeline`, списки столбцов, ID опыта, конфигурация, версии библиотек, версия данных.
- `output/model/input_schema.json` — то же без модели, читаемо без Python: имена и порядок входных столбцов, типы, параметры, seed, версии, контрольные суммы входа.
- `output/validation_predictions.csv` — ключи, score, rank и признак попадания в top-K. Истинной метки в файле нет, она остаётся в `labels.csv`.

Пользовательский модуль нужен только Pipeline регрессии: её подготовка ссылается на функцию из `starter/preprocessing_template.py` по имени модуля. Pipeline бустинга состоит из одних классов scikit-learn. Что требуется выбранному кандидату, записано в `input_schema.json` (`required_module`).

In [17]:
numeric_selected, categorical_selected = columns_of(selected_experiment)
selected_pipeline = PIPELINES[SELECTED]

bundle = {"pipeline": selected_pipeline, "numeric_columns": numeric_selected, "categorical_columns": categorical_selected,
          "selected_experiment": SELECTED, "config": config, "versions": VERSIONS, "data_version": DATA_VERSION}
MODEL_PATH = MODEL_DIR / "candidate.joblib"
joblib.dump(bundle, MODEL_PATH)

schema = {
    "selected_experiment": SELECTED, "family": selected_experiment["family"], "params": selected_experiment["params"],
    "seed": SEED, "feature_set": selected_experiment["feature_set"],
    "input_columns_in_order": [{"name": c, "dtype": str(train["X"][c].dtype), "missing": "NaN"} for c in numeric_selected]
                              + [{"name": c, "dtype": "object (text)", "missing": "NaN"} for c in categorical_selected],
    "score": "predict_proba column of class 1 (pipeline.classes_ == [0, 1])",
    "fitted_on": "train, snapshots 2025-12-01..2026-07-01", "campaign_fraction": FRACTION,
    "versions": VERSIONS, "data_version_sha256": DATA_VERSION,
    "required_module": ("starter/preprocessing_template.py (must be importable as preprocessing_template)"
                        if selected_experiment["family"] == "logistic_regression" else "none: scikit-learn classes only"),
}
(MODEL_DIR / "input_schema.json").write_text(json.dumps(schema, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

predictions = RANKED[(SELECTED, "validation")][KEY + ["score", "rank", "selected"]].reset_index(drop=True)
PREDICTIONS_PATH = OUTPUT / "validation_predictions.csv"
write_csv(predictions, PREDICTIONS_PATH)  # scores are written at full precision
print(f"{MODEL_PATH.as_posix()}: {MODEL_PATH.stat().st_size / 1024:.0f} КБ; прогнозов: {len(predictions)}, в top-K: {int(predictions.selected.sum())}")
predictions.head()

output/model/candidate.joblib: 82 КБ; прогнозов: 1929, в top-K: 192


,customer_id,snapshot_at,score,rank,selected
0,C002279,2026-08-01T00:00:00+03:00,0.182044,1,True
1,C002298,2026-08-01T00:00:00+03:00,0.175068,2,True
2,C002026,2026-08-01T00:00:00+03:00,0.174592,3,True
3,C002454,2026-08-01T00:00:00+03:00,0.172918,4,True
4,C002457,2026-08-01T00:00:00+03:00,0.172918,5,True


### Загрузка в новом процессе без fit

`starter/check_saved_model.py` запускается отдельным процессом Python: он не видит переменных ноутбука, читает только файл модели и таблицы этапа 4. Скрипт сам сравнивает ключи, score (допуск 1e-12), ранги и состав top-K с `validation_predictions.csv` и завершается ошибкой при любом расхождении. Меток он не читает, поэтому метрики пересчитываются здесь — по его выходному файлу.

In [18]:
RELOAD_PATH = OUTPUT / "reload_validation_predictions.csv"
command = [sys.executable, "check_saved_model.py", "--model", str(MODEL_PATH.resolve()), "--prepared", str(PREPARED.resolve()),
           "--out", str(RELOAD_PATH.resolve()), "--expected", str(PREDICTIONS_PATH.resolve())]
completed = subprocess.run(command, cwd=STARTER, capture_output=True, text=True)
print(completed.stdout.strip() or completed.stderr.strip())
record_check("saved Pipeline reloads in a new process and reproduces keys, scores, ranks and top-K without fit",
             completed.returncode == 0, completed.stdout.strip() or completed.stderr.strip()[-300:])

reloaded = pd.read_csv(RELOAD_PATH, sep=";", encoding="utf-8-sig", dtype={"customer_id": str, "snapshot_at": str})
paired = reloaded.merge(predictions, on=KEY, validate="one_to_one", suffixes=("_reloaded", "_saved"))
validation_labels = validation["keys"].assign(target=validation["y"])
reloaded_metrics, _, _ = metric_rows(SELECTED, "validation", reloaded[KEY],
                                     reloaded.merge(validation_labels, on=KEY, validate="one_to_one").target.to_numpy(),
                                     reloaded.score.to_numpy())
saved_metrics = by_date_all[(by_date_all.id == SELECTED) & (by_date_all.part == "validation")].reset_index(drop=True)
record_check("metrics recomputed from the reloaded scores equal the metrics of the training run",
             len(paired) == len(predictions) and reloaded_metrics.equals(saved_metrics),
             f"max |score difference| = {(paired.score_reloaded - paired.score_saved).abs().max():.1e}; "
             f"found {int(reloaded_metrics.hits.sum())} of {int(reloaded_metrics.n_churn.sum())}, K = {int(reloaded_metrics.K.sum())}")

Reloaded without fit; validation scores: 1929; expected match: True
[ok] saved Pipeline reloads in a new process and reproduces keys, scores, ranks and top-K without fit — Reloaded without fit; validation scores: 1929; expected match: True
[ok] metrics recomputed from the reloaded scores equal the metrics of the training run — max |score difference| = 9.9e-17; found 18 of 63, K = 192


### Повторное обучение и запись результатов

Выбранный опыт обучается второй раз с нуля: оценки должны совпасть, иначе результат зависел бы от случайности запуска. Затем записываются журнал, таблицы и контрольные суммы.

Контрольные суммы сравниваются с предыдущим запуском (`output/checksums.csv`). В сравнение не входят файл модели (бинарный формат) и журнал с временем обучения: время от запуска к запуску меняется. Если код или данные менялись намеренно, проверка один раз остановит ноутбук со списком изменившихся файлов; следующий запуск сравнит уже с новой версией.

In [19]:
_, again, _, _ = run(selected_experiment, train, {"validation": validation})
record_check("refitting the selected experiment from scratch gives identical validation scores",
             bool(np.array_equal(again["validation"], SCORES[SELECTED]["validation"])),
             f"{SELECTED}: {len(again['validation'])} scores are bit-identical")

# Experiment log: one row per approach with parameters, timing, status and both sets of metrics.
baseline_row = pd.DataFrame([{"order": 0, "id": "baseline", "method": "Правило: сортировка по долгу",
                              "params": json.dumps({"sort": f"{BASELINE_FEATURE} desc, customer_id asc"}),
                              "x_version": "—", "n_features": 1, "status": "ok", "warnings": ""}])
experiment_log = pd.concat([baseline_row, log], ignore_index=True).merge(results, on="id", validate="one_to_one")
experiment_log["selected"] = experiment_log.id == SELECTED
metric_columns = [c for c in results.columns if c.endswith(("recall", "precision", "ap", "validation"))]
experiment_log[metric_columns] = experiment_log[metric_columns].round(6)

selection = {"selected_experiment": SELECTED, "rule": config["selection_rule"], "decided_by": decided_by,
             "ranking": ranking.id.tolist(),
             "validation": {"date": "2026-08-01", "K": len(candidate_top), "churned": int(row.validation_churn[SELECTED]),
                            "baseline_found": len(baseline_found), "candidate_found": len(candidate_found),
                            "additional_found": len(candidate_found) - len(baseline_found),
                            "found_only_by_candidate": only_candidate, "found_only_by_baseline": only_baseline,
                            "sign_test_p": round(p_value, 4)},
             "temporal_folds": {"check_dates": fold_dates, "churned": int(row.fold_churn[SELECTED]),
                                "baseline_found": int(row.fold_hits["baseline"]),
                                "candidate_found": int(row.fold_hits[SELECTED])},
             "test_used": False, "refit_on_train_plus_validation": False}

write_csv(experiment_log, OUTPUT / "experiment_log.csv")
write_csv(by_date_all.round(6), OUTPUT / "metrics_by_date.csv")
write_csv(fold_by_date.round(6), OUTPUT / "fold_metrics_by_date.csv")
(OUTPUT / "selection.json").write_text(json.dumps(selection, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(OUTPUT / "versions.json").write_text(json.dumps(VERSIONS, indent=2) + "\n", encoding="utf-8")
print("Записано:", ", ".join(sorted(p.relative_to(OUTPUT).as_posix() for p in OUTPUT.rglob("*") if p.is_file())))

[ok] refitting the selected experiment from scratch gives identical validation scores — hgb_leaf4_nodig: 1929 scores are bit-identical
Записано: checks.csv, checksums.csv, experiment_log.csv, fold_metrics_by_date.csv, metrics_by_date.csv, model/candidate.joblib, model/input_schema.json, reload_validation_predictions.csv, selection.json, validation_predictions.csv, versions.json


In [20]:
# versions.json is a record of this machine, not a result: left out so that another machine can reproduce the sums.
NOT_COMPARED = {"checks.csv", "checksums.csv", "experiment_log.csv", "candidate.joblib", "versions.json"}
generated = sorted(p for p in OUTPUT.rglob("*") if p.is_file() and p.name not in NOT_COMPARED)
current = pd.DataFrame({"file": [p.as_posix() for p in generated], "sha256": [sha256(p) for p in generated]})
checksums_path = OUTPUT / "checksums.csv"
if checksums_path.exists():
    previous = read_csv(checksums_path)
    merged = current.merge(previous, on="file", how="outer", suffixes=("", "_previous"), indicator=True)
    changed = merged[(merged._merge != "both") | (merged.sha256 != merged.sha256_previous)].file.tolist()
    record_check("deterministic outputs are byte-identical to the previous run", not changed,
                 f"{len(current)} files; changed: {changed or 'none'}")
else:
    record_check("deterministic outputs are byte-identical to the previous run", True,
                 f"first run: {len(current)} checksums saved, nothing to compare with")
write_csv(current, checksums_path)

checks = pd.DataFrame(CHECKS)
write_csv(checks, OUTPUT / "checks.csv")
failed = checks[~checks.passed]
require(failed.empty, f"{len(failed)} check(s) failed: {failed.check.tolist()}")
print(f"Все проверки пройдены: {len(checks)}")

[ok] deterministic outputs are byte-identical to the previous run — 6 files; changed: none
Все проверки пройдены: 21


*Шаг 7. Сохранение и воспроизводимость*

---

## Итог

| Подход | Train Recall@K | Фолды внутри train, найдено / ушли | Validation, найдено / ушли | Validation Recall@K |
|---|---:|---|---|---:|
| Baseline: сортировка по долгу | 39,6% | 97 / 236 | 14 / 63 | 22,2% |
| Регрессия, лучшая (`lr_c1_nodig`) | 39,6% | 88 / 236 | 16 / 63 | 25,4% |
| Бустинг, выбран (`hgb_leaf4_nodig`) | 45,6% | 92 / 236 | 18 / 63 | 28,6% |

- Два метода обучены на train в шести конфигурациях; подготовка признаков — внутри Pipeline, `fit` только на train.
- Кандидат `hgb_leaf4_nodig` выбран по заранее записанному правилу и сохранён целиком. В новом процессе без `fit` он воспроизвёл те же score, ранги, top-K и метрики.
- На validation кандидат нашёл на 4 ухода больше правила; на проверочных датах внутри train — на 5 меньше. Преимущество не доказано.
- Test 01.09.2026 не использован.

Файлы: `output/experiment_log.csv` (журнал), `output/metrics_by_date.csv` и `output/fold_metrics_by_date.csv` (таблицы по T), `output/validation_predictions.csv`, `output/model/`, `output/selection.json`, `output/checks.csv`. Отчёт и журнал в читаемом виде — `docs/training_report.md` и `docs/experiment_journal.md`.